# EXP-005 — M4 CPG-GNN: bước 1, TRÍCH CPG bằng Joern (CPU)

Notebook này **không train**. Nó dựng CPG cho từng hàm và — quan trọng hơn —
**đo throughput thật** và **kiểm tra CPG có đủ loại cạnh không**, để quyết định
M4 có đáng làm không.

**Vì sao cài Joern ở đây mà không cài ở máy?** Máy dự án không có Java; Kaggle có
sẵn JDK 17 và quota CPU miễn phí. Cài local sẽ tốn hàng trăm MB, hỏng môi trường,
và không giúp gì — vì **train vẫn phải chạy trên Kaggle**.

**Sửa đúng 1 chỗ:** cell 2 — dòng `REPO = "<user>/<repo>"`.

### Thứ tự chạy (đừng bấm Run All)
1. Cell 2 (**clone hoặc pull** code mới) → cell 3 (cài JDK + Joern)
2. Cell 4 (SMOKE) → cell 5 (API SCAN) → cell 6 (dataset)
3. Cell 7: **đo 20 hàm** — có gate cứng, 0 JSON là **dừng luôn**
4. Cell 8: đo 500 hàm (chỉ chạy được sau khi cell 7 qua)
5. Cell 9: nạp cache + **gate chất lượng thật, bằng assert**
6. **Save Version** → đó là input cho notebook train M4 (bước 2)

### Cell 5 (API SCAN) là bắt buộc, không phải tuỳ chọn
Tên step CPGQL **không đoán được**: Joern v4 đã bác nhiều tên mà bảng `.help`
vẫn liệt kê. Rõ nhất là `call.reachingDef` — **không tồn tại**, cả dạng không tham
số lẫn dạng `reachingDef(false)` đều `Not Found` (không phải "thiếu tham số").
Bảng `.help` vì thế **không phải** bằng chứng; compiler mới là bằng chứng.
Cell 5 chạy từng biểu thức một tiến trình Joern riêng và ghi kết quả vào
`docs/joern_api_v4.json`; exporter chỉ được dùng step đã xác nhận. Kết quả được
cache nên chạy lại chỉ tốn thời gian cho candidate mới (~13s mỗi cái).

### Ba sự thật đã biết — đừng phát hiện lại
- **`n_edges` lớn không bảo chứng có CPG.** Một đồ thị đầy AST+CFG có
  `edges_per_code_mean` rất cao mà vẫn không có data-flow. Cell 9 kiểm tra
  `edge_types_present`, không kiểm tra `n_edges`.
- **DEC-027 (đã chốt): M4 chạy phương án (b).** Cell 5 đã chứng minh mọi đường
  đọc cạnh data-flow qua DSL đều FAIL (`call.reachingDef*`, `cpg.graph.E*`,
  `call.ref`...), nên `CALL`/`REF`/`REACHING_DEF` ra khỏi hợp đồng schema
  (`EDGE_LABELS` còn 6 loại: AST, CFG, ARGUMENT, TARG, PARAMETER_OUT,
  EVAL_ORDER). TARG (kết quả gọi hàm → biến đích, nội hàm) là tín hiệu
  data-flow duy nhất giữ được.
- **Lô chết sẽ tự in dòng lỗi thật** Joern báo (grep `E008`/exception, kèm ngữ
  cảnh) — không phải 12 dòng banner khởi động. Log đầy đủ vẫn nằm trong
  `artifacts/cpg_work/joern_batch_*.log` nếu cần.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

# Cách 1 (khuyên dùng): Kaggle Add-ons → Secrets → GITHUB_TOKEN + HF_TOKEN
# Cách 2: dán trực tiếp dưới đây (ĐỪNG share notebook này)
GITHUB_TOKEN = ""
HF_TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    GITHUB_TOKEN = GITHUB_TOKEN or _s.get_secret("GITHUB_TOKEN")
    HF_TOKEN = HF_TOKEN or _s.get_secret("HF_TOKEN")
except Exception as _e:
    print("[secrets] khong doc duoc secrets:", _e)
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
if GITHUB_TOKEN:
    os.environ.setdefault("GIT_TERMINAL_PROMPT", "0")
print("[secrets] github =", bool(GITHUB_TOKEN), "| hf =", bool(HF_TOKEN))

def sh(cmd, **kw):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    print(f"$ {cmd}\n{r.stdout}{r.stderr}")
    if r.returncode != 0:
        raise RuntimeError(f"lenh that bai (rc={r.returncode}): {cmd}")
    return r.stdout

def py(code, **kw):
    return sh(sys.executable.replace('"', '') + " -c " + json.dumps(code), **kw)


# --- Gate dùng lại cho cell 7/8/9 -------------------------------------
# Mỗi lần gọi Joern tốn ~1–2 phút. Chạy tiếp các cell sau khi lô trước đã
# chết chỉ tạo thêm cache 0 đồ thị rồi lại phải đoán, nên ở đây **dừng thật**.
def show_bench(path):
    d = json.load(open(path, encoding='utf-8'))
    for k in ('n_requested', 'n_ok', 'n_error', 'n_missing', 'error_rate',
              'functions_per_second', 'seconds_per_function', 'wall_seconds',
              'nodes_per_cpg_mean', 'json_bytes_mean', 'est_hours_full_dataset'):
        print(f'  {k:26s} {d.get(k)}')
    print('  edge_types_present        ', d.get('edge_types_present'))
    print('  edge_types_never_produced ', d.get('edge_types_never_produced'))
    return d


def require_ok(path):
    d = show_bench(path)
    assert d['n_ok'] > 0, (
        f"0/{d['n_requested']} ham ra JSON — DUNG tai day, khong chay tiep. "
        "Doc khoi '[cpg-export] LOT CHET' o phia tren: phan lon la loi compile "
        "Scala, va dong do da nam san trong output roi.")
    return d

In [ ]:
REPO = "kichirou244/SemanticCodeCloneDetection"
assert REPO != "YOUR_USER/YOUR_REPO", "Sua REPO truoc khi chay cell tiep theo"

if Path(".git").exists():
    # Đã clone ở session trước ⇒ **phải pull**. Không pull thì chạy code cũ và
    # đi tìm lỗi đã sửa xong từ lâu — đã mất nhiều vòng Joern vì chuyện này.
    sh('git pull --ff-only || (git fetch --all --depth 1 && '
       'git reset --hard origin/HEAD)')
else:
    url = f'https://github.com/{REPO}.git'
    if GITHUB_TOKEN:
        url = url.replace('https://', f'https://x-access-token:{GITHUB_TOKEN}@')
    sh(f'git clone --depth 1 {url} repo')
    sh('mv repo/* repo/.[!.]* . 2>/dev/null; rmdir repo')
sh('git log --oneline -1 && python -V && nproc && free -g | head -2')

# CPG không đụng torch: notebook này chỉ trích xuất (CPU). torch là của
# notebook train M4 — cài ở đây sẽ tốn thời gian vô ích. Cũng không cài
# torch-geometric: tầng model chưa chạy ở notebook này.
import importlib
need = []
for mod, pkg in [('numpy', 'numpy'), ('yaml', 'pyyaml'), ('tqdm', 'tqdm')]:
    try:
        importlib.import_module(mod)
    except ImportError:
        need.append(pkg)
print('can cai:', need or 'khong')
if need:
    sh('pip install -q ' + ' '.join(need))
print(sh('python -c "import src.representations.cpg.schema as s; '
           'print(s.NODE_LABELS); print(s.EDGE_LABELS)"'))

In [ ]:
# CÀI JDK 17 + JOERN — chỉ ở đây (Kaggle), KHÔNG cài máy local.
sh("apt-get update -qq && apt-get install -y -qq openjdk-17-jre-headless")
sh("java -version 2>&1 | head -3")

# Tìm Joern đã có, theo đúng nơi installer chính thức đặt binary. PHẢI thử nhiều
# đường: script cài chọn `~/bin/joern` khi chạy bình thường, nhưng khi notebook
# chạy dưới root thì nó đặt vào `/opt/joern` + symlink vào `/usr/local/bin`.
# Chỉ check một đường là fail giả, rồi tải lại 1,7GB cho nothing.
import shutil

def find_joern():
    for p in (Path.home() / 'bin' / 'joern', Path('/opt/joern/joern'),
              Path('/usr/local/bin/joern')):
        if p.exists():
            return p
    w = shutil.which('joern')
    return Path(w) if w else None

JOERN_BIN = find_joern()

if JOERN_BIN is None:
    # Chưa có thì mới cài. Dùng script chính thức: tự dò đúng bản theo platform,
    # và KHÔNG hardcode tên file zip vì Joern phát hành bản mới hằng ngày, tên
    # asset đổi theo version (v4.x là joern-cli-linux-x86_64.zip).
    sh('wget -q https://github.com/joernio/joern/releases/latest/download/'
       'joern-install.sh -O /tmp/joern-install.sh && '
       'chmod +x /tmp/joern-install.sh && /tmp/joern-install.sh')
    JOERN_BIN = find_joern()

assert JOERN_BIN is not None, (
    'Khong tim thay joern. Neu vua cai xong, tim bang lenh: '
    '`ls -l /usr/local/bin/joern* /opt/joern/joern*` roi dan duong dan vao JOERN_BIN')
os.environ['JOERN_BIN'] = str(JOERN_BIN)
print('[joern] binary =', JOERN_BIN)

# Gate: phải cả java lẫn joern mới chạy tiếp. In báo cáo rõ ràng thay vì để
# Joern chết âm thầm ở cell sau.
import json as _json
from src.representations.cpg.build import cpg_status
st = cpg_status()
print(_json.dumps(st, indent=2, ensure_ascii=False))
assert st['ready'], 'Thieu java/joern — dung cell nay truoc khi chay tiep'

In [ ]:
# SMOKE — Joern thật trên đúng 1 hàm, in ra ngay.
#
# Mục đích: chứng minh Joern **parse được code C++** trên môi trường này. Còn
# tên step CPGQL thì cell 5 lo — đừng kỳ vọng cell 4 trả lời chuyện đó, vì nó
# không chạy exporter nên không thể biết exporter có chạy được hay không.
# Đọc kết quả:
#   - có "=== METHODS ===" và các dòng ast=/argument=/cfgNode=...  → OK, đi tiếp
#   - lỗi compile Scala hoặc thiếu dòng nào                       → DỪNG, gửi outputsh('python scripts/build_cpg_export.py --config configs/exp005_cpg_gnn.yaml --smoke')

In [ ]:
# API SCAN — chốt tên step CPGQL bằng compiler, không đoán và không tin `.help`.
#
# Mỗi ứng viên là một tiến trình Joern riêng: nếu gộp vào một file thì một
# dòng sai làm hỏng cả nhóm, và ta mất thông tin về các ứng viên còn lại.
# Kết quả ghi vào docs/joern_api_v4.json và **được cache** — lần sau chỉ chạy
# ứng viên mới (~13s mỗi cái).
#
# Dòng "=== KHONG HOP LE ===" là phần quan trọng nhất: đó là danh sách tên
# exporter được phép động tới. Nếu một tên nằm trong đó mà vẫn xuất hiện trong
# export_cpg.sc thì exporter sẽ chết lúc compile.
print(sh('python scripts/joern/api_scan.py'))

In [ ]:
sh('python scripts/build_dataset.py --config configs/pairs/poj104.yaml')
py("import json; m=json.load(open('data/manifests/poj104.json')); "
   'print(\"leakage_passed =\", m[\"leakage_passed\"]); '
   'print(\"pairs =\", {k:v for k,v in m.get(\"pairs\",{}).items()})')
sh('wc -l data/processed/poj104/samples.jsonl')


In [ ]:
# ĐO THỬ 20 HÀM — phép đo quyết định M4 có đáng làm không.
#
# Nếu `functions_per_second` < ~0,3 hàm/s thì 52k hàm > 1 ngày ⇒ dừng và bàn
# lại phạm vi trước khi đốt quota Kaggle. `est_hours_full_dataset` in ra là
# con số cần đọc, không phải chi tiết phụ.
#
# Xoá thư mục cũ trước: nếu lần trước hỏng thì còn sót `benchmark.json` của
# lần hỏng, và đọc nhầm là đo xong.
sh('rm -rf artifacts/cpg_probe')
sh('python scripts/build_cpg_export.py --config configs/exp005_cpg_gnn.yaml '
   '--split train --limit 20 --batch-size 20 --out-dir artifacts/cpg_probe')
bm20 = require_ok('artifacts/cpg_probe/benchmark.json')

# Xem thử một hàm thật để chắc JSON đọc được và id là cục bộ (0..n-1), không
# phải id nội bộ của Joern (~1e11, dễ tràn int32 khi đưa vào PyG).
one = sorted(p for p in Path('artifacts/cpg_probe').glob('*.json')
            if p.name != 'benchmark.json')
assert one, 'khong co file .json nao duoc tao ra'
sample = json.load(open(one[0], encoding='utf-8'))
ids = [n['id'] for n in sample['nodes']]
print('file =', one[0].name)
print('n_nodes =', len(ids), '| id range =', min(ids), '..', max(ids))
print('labels =', sorted({n['label'] for n in sample['nodes']}))
print('edges  =', sorted({e['label'] for e in sample['edges']}))
print('n_args co tren node CALL =',
      sum(1 for n in sample['nodes'] if n['label'] == 'CALL' and 'n_args' in n))

In [ ]:
# ĐO 500 HÀM — con số dùng để chốt ngân sách giờ cho M4 (thay ước lượng
# 0,3–1 hàm/s trong reports/005). Vài phút, chạy trước khi cam kết cả giờ.
#
# Gate: chặn nếu 20 hàm ở cell 7 chưa ra JSON. Vì mỗi lô Joern tốn phút, chạy
# tiếp khi lô trước đã chết là đốt quota để rồi vẫn ra cache rỗng.
require_ok('artifacts/cpg_probe/benchmark.json')
sh('rm -rf artifacts/cpg_export')
sh('python scripts/build_cpg_export.py --config configs/exp005_cpg_gnn.yaml '
   '--split train --limit 500 --batch-size 100 --out-dir artifacts/cpg_export')
bm500 = require_ok('artifacts/cpg_export/benchmark.json')

h = bm500.get('est_hours_full_dataset')
print(f'\n[uoc luong] 500 ham -> {h} gio cho ca dataset')
print('[uoc luong] con bao nhieu gio quota: xong o day, dung chay tiep neu con it.')

In [ ]:
# Nạp cache .npz từ JSON vừa có (nạp 1 lần, các lần sau chỉ đọc cache).
# max_nodes lấy đúng từ config — đổi số này là đổi feature, cache cũ sẽ sai.
# Xoá cache cũ: nếu để lại thì cell này đọc cache cũ và báo số của phiên trước.
require_ok('artifacts/cpg_export/benchmark.json')
sh('rm -f artifacts/cpg_cache/cpp_train500.npz')
from src.common.config import load_config
from src.representations.cpg.store import CPGStore
from src.representations.cpg.schema import EDGE_LABELS

_pre = load_config(Path('configs/exp005_cpg_gnn.yaml'))['preprocessing']
_ids = [json.loads(l)['sample_id'] for l in
        open('data/processed/poj104/samples.jsonl', encoding='utf-8')
        if json.loads(l)['split'] == 'train'][:500]
st = CPGStore.build(Path(_pre['export_dir']), _ids, _pre['max_nodes'],
                   Path(_pre['cache_dir']) / 'cpp_train500.npz').stats
print(json.dumps({k: v for k, v in st.items()
                  if k not in ('missing', 'failed')},
                 indent=2, ensure_ascii=False))

# --- GATE CHẤT LƯỢNG: assert thật, không phải comment để mắt đọc ----------
# Đọc `edge_type_counts` (do `graph_stats()` sinh ra), KHÔNG phải
# `edge_types_present` — cái đó là tên của `benchmark.json`, hai bên khác
# nhau. Đọc nhầm thì `present` rỗng và gate báo "thiếu AST/CFG" trong lúc
# `edge_type_counts` ngay dòng trên còn in ra đủ 3 loại cạnh.
present = set(st.get('edge_type_counts') or {})
missing = [k for k in EDGE_LABELS if k not in present]
print('\n=== GATE ===')
print('edge_types_present  =', sorted(present))
print('edge_types_missing  =', missing)
print('nodes_per_code_mean =', st.get('nodes_per_code_mean'))
print('edges_per_code_mean =', st.get('edges_per_code_mean'))

assert st['n_loaded'] > 0, 'cache rong — 0 do thi nen khong co gi de kiem tra'
assert st.get('nodes_per_code_mean', 0) >= 20, (
    f"nodes_per_code_mean={st.get('nodes_per_code_mean')} < 20 => Joern khong "
    'ra CPG that. Dung M4, sua exporter.')
assert st['n_missing'] / max(st['n_loaded'] + st['n_missing'], 1) <= 0.05, (
    f"n_missing={st['n_missing']} > 5% => parse hong nhieu, phai xem tung ham")
assert {'AST', 'CFG'} <= present, (
    f'thetu AST/CFG ({sorted(present)}) — khong con la CPG')
print('\nGate AST+CFG: PASS')
# DEC-027: REACHING_DEF/REF/CALL da ra khoi hop dong schema — khong kiem nua.
# Thu duy nhat con theo doi o day la PARAMETER_OUT: dang 0/500 do bug so
# identity trong exporter, cho fix khop ten sau khi cell 5 xong vong 6
# (parameter.name/code, argument.code). Chua fix thi chua Save.
if 'PARAMETER_OUT' in missing:
    print('[CANH BAO] PARAMETER_OUT van 0 — cho fix khop ten (cell 5 vong 6).')
    print('KHONG Save Version.')
else:
    print('Gate PARAMETER_OUT: PASS.')

### Save Version — nhưng **chỉ** khi cell 9 in `Gate PARAMETER_OUT: PASS`

Version đã lưu **là input** cho notebook train M4 (đọc `artifacts/cpg_export`
và `artifacts/cpg_cache`). Không Save Version thì mất toàn bộ CPG đã dựng.

Nếu cell 9 in cảnh báo `PARAMETER_OUT` vẫn 0 thì **dừng lại**, chưa Save: chờ
fix khớp tên (sau khi cell 5 xong vòng 6) rồi chạy lại extraction. DEC-027
(`REACHING_DEF`/`REF`/`CALL` ra khỏi hợp đồng) đã chốt, không còn gì để quyết
ở đây.

**Trước khi Save, gửi lại 5 con số** (chúng quyết định ngân sách M4):
1. `functions_per_second` (cell 8)
2. `est_hours_full_dataset` (cell 8)
3. `error_rate` (cell 8)
4. `nodes_per_code_mean` (cell 9)
5. `edge_types_present` + `edge_types_missing` (cell 9)

Kèm theo nếu có: `docs/joern_api_v4.json` từ cell 5 (danh sách step Joern v4
thực sự chấp nhận, gồm vòng 6 cho fix `PARAMETER_OUT`), và output cell 4
(smoke).

### Chưa làm gì ở bước này (cố ý)
- **Chưa chạy cả split:** `est_hours_full_dataset` là quyết định ngân sách,
  không phải việc làm mù. Còn bao nhiêu giờ quota thì đo trước đã.
- **Chưa chạy nhiều tiến trình song song:** giữ nguyên 1 tiến trình để số đo
  đáng tin; song hoá là việc làm sau, khi đã biết số đo một tiến trình.
- **Chưa train GAT:** cần số đo trước (DEC-024).